In [1]:
# Importamos las librerías básicas

import numpy as np
import pandas as pd

# modificamos la configuración para ver todas las columnas al mostrar dataframes
pd.set_option("display.max_columns", None)

In [2]:
# Cargamos datos limpiosde nuestro primer dataset
bank_df = pd.read_csv ('../Data/DataProcessed/bank-additional-clean.csv', sep=",")
# Hacemos una copia para trabajar con ella
df_bank = bank_df.copy()

Creamos la función null_values para poder reutilizarla en cualquier dataframe

In [3]:
def null_values(dataframe, only_nulls: bool = False):
    """
    Genera un DataFrame con el conteo y porcentaje de valores nulos por columna.
    
    Parámetros:
    -----------
    dataframe : pd.DataFrame
        DataFrame de entrada.
    only_nulls : bool, opcional (default=False)
        Si True, devuelve únicamente las columnas con al menos un nulo.
    
    Devuelve:
    ---------
    df_null : pd.DataFrame
        DataFrame con columnas:
            - null_counts : número de nulos
            - null_% : porcentaje de nulos sobre el total
    """
    null_counts = dataframe.isnull().sum()
    null_percent = (null_counts / dataframe.shape[0] * 100).round(2)
    df_null = pd.DataFrame({
        "null_counts": null_counts,
        "null_%": null_percent
    })

    if only_nulls:
        df_null = df_null[df_null["null_counts"] > 0]
    
    return df_null


In [4]:
df_null = null_values(df_bank)
#Para obtener un resumen ordenado de mayor a menor de nulos
display(df_null.sort_values("null_%", ascending=False))

,null_counts,null_%
euribor3m,9256,21.53
default,8981,20.89
age,5120,11.91
education,1807,4.20
loan,1026,2.39
housing,1026,2.39
cons.price.idx,471,1.10
job,345,0.80
contact_year,248,0.58
contact_month,248,0.58


In [5]:
# Solo que muestre columnas con nulos
df_null= null_values(df_bank, only_nulls=True)
display(df_null.sort_values("null_%", ascending=False))

,null_counts,null_%
euribor3m,9256,21.53
default,8981,20.89
age,5120,11.91
education,1807,4.20
loan,1026,2.39
housing,1026,2.39
cons.price.idx,471,1.10
job,345,0.80
contact_month,248,0.58
date,248,0.58


### Vamos a tratar los nulos en el orden de la lista anterior, es decir, primero trataremos aquellos con mayor porcentaje de nulos

In [6]:
#Obtenemos un dataframe que solo tiene valores nulos de euribor3m
df_bank_euribornull = df_bank[df_bank['euribor3m'].isna()]
df_bank_euribornull.sample(5)

,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_,contact_month,contact_year
36344,33.0,blue-collar,MARRIED,professional.course,no,NaN,NaN,cellular,240,1,999,0,NONEXISTENT,-2.9,92.963,-40.8,NaN,5076,no,2015-09-03,48.760,-101.157,98d44a26-7700-4f56-8756-090069edf420,September,2015.0
4417,27.0,admin.,MARRIED,high.school,no,yes,no,telephone,210,2,999,0,NONEXISTENT,1.1,93.994,-36.4,NaN,5191,no,2018-11-30,27.407,-87.757,dea70596-83a2-43d7-b341-ca7c19a97443,November,2018.0
21166,51.0,admin.,MARRIED,university.degree,NaN,yes,no,cellular,204,5,999,0,NONEXISTENT,1.4,93.444,-36.1,NaN,5228,no,2015-11-08,48.269,-101.994,03fc8d4c-48b0-4178-96b9-5a6684f1dd76,November,2015.0
1621,32.0,services,MARRIED,high.school,no,no,no,telephone,213,2,999,0,NONEXISTENT,1.1,93.994,-36.4,NaN,5191,no,2017-03-08,32.797,-89.007,5c71250e-9787-4ffc-b1bd-eae957f00fd9,March,2017.0
29353,34.0,admin.,SINGLE,university.degree,no,yes,no,cellular,285,1,999,0,NONEXISTENT,-1.8,93.075,-47.1,NaN,5099,no,2015-09-13,43.701,-85.449,0bf7f5cc-7da2-4f7e-abea-d383889175bb,September,2015.0


In [8]:
df_bank_euribornull['date'].unique

<bound method Series.unique of 1        2016-09-14
3        2015-11-29
4        2017-01-29
9        2016-11-02
11       2016-04-17
            ...    
42992    2016-03-23
42993    2018-04-04
42995    2015-10-13
42996    2018-03-17
42997    2016-09-15
Name: date, Length: 9256, dtype: object>

In [32]:
df_bank[df_bank['date']=='2017-01-29'].sample(5)

,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_,contact_month,contact_year
16251,53.0,admin.,SINGLE,NaN,NaN,yes,no,cellular,319,2,999,0,NONEXISTENT,1.4,93.918,-42.7,4.961,5228,no,2017-01-29,30.658,-73.132,8ea0436c-9b19-46af-bb45-724a021369af,January,2017.0
19530,50.0,technician,MARRIED,professional.course,no,no,no,cellular,139,2,999,0,NONEXISTENT,1.4,93.444,-36.1,NaN,5228,no,2017-01-29,29.894,-105.792,90c7eb66-4a75-4ece-ab25-4389a8d7928e,January,2017.0
4,56.0,services,MARRIED,high.school,no,no,yes,telephone,307,1,999,0,NONEXISTENT,1.1,93.994,-36.4,NaN,5191,no,2017-01-29,38.033,-104.463,eca60b76-70b6-4077-80ba-bc52e8ebb0eb,January,2017.0
41579,45.0,management,SINGLE,basic.9y,no,yes,no,cellular,165,3,999,1,FAILURE,-0.1,93.200,-42.0,4.076,5195,no,2017-01-29,32.202,-120.791,0f41f00f-ed15-4e4e-b04f-c53b3feaec29,January,2017.0
27985,37.0,technician,DIVORCED,basic.9y,no,yes,no,cellular,247,1,999,1,FAILURE,-1.8,93.075,-47.1,1.466,5099,no,2017-01-29,27.832,-101.621,ce4e5417-36da-4cd2-b1c3-935b55140731,January,2017.0


Hemos obtenido un data frame que contiene únicamente los registros con un valor de euribor3m nulo para realizar una inspección. Hemos identificado diferentes fechas que tienen valores nulos de euribor3m asociados. Hemos vuelto al data frame original para buscar algunas de estas fechas y ver si nos pueden dar una estmiación del euribor3m en ese momento pero el propio dataframe presenta diferentes valores de euribor3m para una misma fecha lo que creo que es una insconsistencia.

Para cada fecha, vemos cuántos valores diferentes de euribor3m existen:

In [9]:
df_bank.groupby('date')['euribor3m'].nunique().sort_values(ascending=False)


date
2016-02-28    44
2017-02-28    44
2018-02-28    37
2015-02-28    36
2019-02-28    35
              ..
2019-08-10     6
2016-09-28     6
2017-02-18     6
2018-10-23     6
2015-09-17     5
Name: euribor3m, Length: 1825, dtype: int64

Comprobamos que efectivamente hay distintos valores de euribor3m para una misma fecha

Vamos a utilizar la media por fecha

Para cada fecha, reemplazaremos los valores nulos por la media 
de los valores existentes para esa fecha.

In [12]:
def rellenar_euribor3m_por_fecha(df, col_date='date', col_euribor='euribor3m'):
    """
    Rellena los valores nulos de la columna euribor3m usando la media de la misma fecha.

    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame original con los datos.
    col_date : str
        Nombre de la columna que contiene la fecha (por defecto 'date').
    col_euribor : str
        Nombre de la columna de euribor a rellenar (por defecto 'euribor3m').

    Devuelve:
    ---------
    df_rellenado : pd.DataFrame
        DataFrame con los valores nulos de euribor3m rellenados.
    """
    # Nos aseguramos de que la columna de fecha es datetime
    df[col_date] = pd.to_datetime(df[col_date])

    # Calculamos la media del euribor3m por fecha
    euribor_por_fecha = df.groupby(col_date)[col_euribor].mean()

    # Rellenamos los nulos usando la media de la fecha
    df_rellenado = df.copy()
    df_rellenado[col_euribor] = df_rellenado[col_euribor].fillna(df_rellenado[col_date].map(euribor_por_fecha)).round(3)

# Si euribor3m es nulo → se reemplaza por la media de la fecha.
# Si euribor3m tiene valor → se mantiene ese valor.

    return df_rellenado


In [13]:
# Llamamos a la función
df_bank_euribor3m_rellenado = rellenar_euribor3m_por_fecha(df_bank)

# Verificamos que ya no hay nulos en euribor3m
print(df_bank_euribor3m_rellenado['euribor3m'].isnull().sum())


57


Hemos pasado de 9256 a 57 nulos. Puede que los que no se hayan rellenado sea porque no tienen ninguna fecha asociada. Vamos a comprobarlo

In [14]:
df_bank_euribor3m_rellenado[df_bank_euribor3m_rellenado['euribor3m'].isna()].sample(5)

,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_,contact_month,contact_year
10914,42.0,admin.,MARRIED,basic.6y,no,yes,no,telephone,300,3,999,0,NONEXISTENT,1.4,94.465,-41.8,NaN,5228,no,NaT,29.569,-121.643,c8c8b679-30d6-4b60-831c-c062e07efdfc,NaN,NaN
6283,NaN,admin.,MARRIED,university.degree,no,no,no,telephone,160,3,999,0,NONEXISTENT,1.1,93.994,-36.4,NaN,5191,no,NaT,40.345,-102.903,028e2073-5207-47d1-bd2e-c723e35af18b,NaN,NaN
23720,NaN,technician,SINGLE,university.degree,no,yes,no,cellular,204,1,999,0,NONEXISTENT,1.4,93.444,-36.1,NaN,5228,no,NaT,32.259,-123.482,b5db41ba-89d0-4dda-b5d2-6664464cbf9e,NaN,NaN
2564,55.0,blue-collar,MARRIED,basic.9y,no,yes,no,telephone,323,4,999,0,NONEXISTENT,1.1,93.994,-36.4,NaN,5191,no,NaT,36.111,-98.368,4953216b-51dc-4615-bb94-2100a841569c,NaN,NaN
9781,38.0,blue-collar,MARRIED,basic.6y,no,yes,no,telephone,362,1,999,0,NONEXISTENT,1.4,94.465,-41.8,NaN,5228,no,NaT,40.139,-98.442,16c37238-468c-42b9-b79f-0a29f2b8a85d,NaN,NaN


Comprobado

## Nulos de la columna default

In [47]:
#Obtenemos un dataframe que solo tiene valores nulos de euribor3m
df_bank_default_null = df_bank[df_bank['default'].isna()]
df_bank_default_null.sample(5)

,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_,contact_month,contact_year
9211,49.0,housemaid,MARRIED,basic.4y,NaN,yes,no,telephone,190,2,999,0,NONEXISTENT,1.4,94.465,-41.8,4.967,5228,no,NaT,30.545,-99.456,cd2797a1-23b4-45d9-b8c8-7bd3621608a9,NaN,NaN
13176,41.0,blue-collar,SINGLE,basic.4y,NaN,NaN,NaN,cellular,72,1,999,0,NONEXISTENT,1.4,93.918,-42.7,4.962,5228,no,2015-03-02,36.081,-72.048,83594780-be59-4b35-aa83-cf7451526733,March,2015.0
14101,40.0,blue-collar,MARRIED,basic.6y,NaN,yes,no,cellular,127,3,999,0,NONEXISTENT,1.4,93.918,-42.7,4.962,5228,no,2016-08-16,44.345,-97.654,08fb4154-77df-45f2-98de-759f667a4cb7,August,2016.0
26722,33.0,admin.,MARRIED,university.degree,NaN,yes,no,cellular,386,2,999,0,NONEXISTENT,-0.1,93.200,-42.0,4.076,5195,no,2017-10-29,26.853,-79.925,08624db5-20d9-4fdf-a7b0-436545495ac5,October,2017.0
19474,36.0,admin.,MARRIED,university.degree,NaN,no,no,cellular,90,1,999,0,NONEXISTENT,1.4,93.444,-36.1,4.968,5228,no,2017-08-04,24.425,-70.932,1325df6b-cc03-4785-a2f4-b5aa1eba4167,August,2017.0


Tenemos un gran porcentaje de nulos (20.89%) y cómo no vemos ninguna relación directa de la columna default con las otras, aparentemente no podemos obtener información de si un cliente tiene algún historial de incumplimiento de pagos a no ser que hagamos un análisis más avanzado y decidamos utilizar un modelo de machine learning, entrenarlo y así predecir estos nulos. La solución más sensata es mantener nulos como categoría “unknown” para no perder información ni asumir algo incorrecto dado que posteriormente queremos realizar un EDA.

In [16]:
def rellenar_con_unknown(df, col_name):
    """
    Rellena los valores nulos de una columna con la categoría 'unknown'.

    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame original con los datos.
    col_name : str
        Nombre de la columna en la que se quieren rellenar los nulos.

    Devuelve:
    ---------
    df_filled : pd.DataFrame
        DataFrame con los nulos de la columna especificada rellenados con 'unknown'.
    """
    df_filled = df.copy()
    df_filled[col_name] = df_filled[col_name].fillna('unknown')
    return df_filled

In [67]:
# Rellenamos la columna 'default'
df_bank_default = rellenar_con_unknown(df_bank, 'default')

# Verificamos
print(df_bank_default['default'].isnull().sum())
print(df_bank_default['default'].value_counts())

0
default
no         34016
unknown     8981
yes            3
Name: count, dtype: int64


Comprobamos que hemos rellenado todos los nulos con unknown y también que para la columna default solo hay 3 registros que tienen 'yes', es decir, que solo 3 clientes tienen algún historial de incumplimiento de pagos. Podríamos pensar que seria correcto rellenar los unknown con un 'no' porque segun los datos la probabilidad de que un registro nulo sea 'yes' es muy baja pero si los nulos no son aleatorios podríamos introducir sesgo.

### Creamos nuestro dataframe que contiene el manejo de todos los valores nulos de las distintas columnas

## Creamos función de manejo de nulos que iremos actualizando

In [20]:
def manejo_nulos(df):
    """
    Función general para manejar nulos en el DataFrame.
    
    Transformaciones:
    -----------------
    1. Rellena euribor3m con la media por fecha.
    2. Rellena la columna default con 'unknown'.
    3. (Opcional) Se pueden añadir más columnas categóricas a rellenar con 'unknown'.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame original con nulos.
    
    Devuelve:
    ---------
    df_clean : pd.DataFrame
        DataFrame con los nulos manejados y listo para análisis.
    """
    # Rellenamos euribor3m con la media por fecha
    df_clean = rellenar_euribor3m_por_fecha(df)
    
    # Rellenamos default con 'unknown'
    df_clean = rellenar_con_unknown(df_clean, 'default')
        
    return df_clean


In [22]:
# Aplicar el manejo de nulos y guardar en dt_bank_sin_nulos
df_bank_sin_nulos = manejo_nulos(df_bank)

# Verificar que no quedan nulos en las columnas transformadas
print(df_bank_sin_nulos[['euribor3m', 'default']].isnull().sum())

euribor3m    57
default       0
dtype: int64
